In [1]:
import pandas as pd
import numpy as np
import os
from census import Census
import us
import time
from dotenv import load_dotenv

**Set up Census API Access**

In [2]:
load_dotenv()  # reads .env into the environment

True

In [3]:
CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")
if not CENSUS_API_KEY:
    raise RuntimeError(
        "No Census API key found. Create a .env file in this script's "
        "directory containing CENSUS_API_KEY=your_key_here, or export "
        "CENSUS_API_KEY as an environment variable before running this "
        "script. Get a free key at https://api.census.gov/data/key_signup.html"
    )

STATE_FIPS = us.states.CA.fips  # '06'

In [4]:
# Full range to capture: 2018-2026 inclusive.
DATA_YEARS = list(range(2018, 2027))  # 2018, 2019, ..., 2026

LATEST_PUBLISHED_VINTAGE = 2024

**Define the Coverage Matrix Path and san diego zip codes**

In [5]:
_COVERAGE_MATRIX_PATH = "TemporalCoverage_Matrix.csv"

In [6]:
san_diego_zips_standardized = [
    "91901", "91902", "91903", "91905", "91906", "91908", "91909", "91910",
    "91911", "91912", "91913", "91914", "91915", "91916", "91917", "91921",
    "91931", "91932", "91933", "91934", "91935", "91941", "91942", "91943",
    "91944", "91945", "91946", "91950", "91951", "91962", "91963", "91976",
    "91977", "91978", "91979", "91980", "92003", "92004", "92007", "92008",
    "92009", "92010", "92011", "92013", "92014", "92018", "92019", "92020",
    "92021", "92022", "92023", "92024", "92025", "92026", "92027", "92028",
    "92029", "92030", "92033", "92036", "92037", "92038", "92039", "92040",
    "92046", "92049", "92051", "92052", "92054", "92055", "92056", "92057",
    "92058", "92059", "92060", "92061", "92064", "92065", "92066", "92067",
    "92068", "92069", "92070", "92071", "92072", "92074", "92075", "92078",
    "92079", "92081", "92082", "92083", "92084", "92085", "92086", "92088",
    "92091", "92092", "92093", "92101", "92102", "92103", "92104", "92105",
    "92106", "92107", "92108", "92109", "92110", "92111", "92112", "92113",
    "92114", "92115", "92116", "92117", "92118", "92119", "92120", "92121",
    "92122", "92123", "92124", "92126", "92127", "92128", "92129", "92130",
    "92131", "92136", "92137", "92138", "92139", "92140", "92142", "92143",
    "92145", "92149", "92150", "92153", "92154", "92159", "92160", "92163",
    "92165", "92166", "92167", "92168", "92169", "92170", "92171", "92172",
    "92173", "92174", "92175", "92176", "92177", "92178", "92186", "92191",
    "92192", "92193", "92195", "92196", "92198",
]

In [7]:
if os.path.exists(_COVERAGE_MATRIX_PATH):
    _cov = pd.read_csv(_COVERAGE_MATRIX_PATH, dtype={"Zip Code": str})
    san_diego_zips_standardized = sorted(_cov["Zip Code"].str.zfill(5).unique())
    print(f"Loaded {len(san_diego_zips_standardized)} zips from {_COVERAGE_MATRIX_PATH}")
else:
    print(f"{_COVERAGE_MATRIX_PATH} not found — using the hardcoded 165-zip fallback list")

sd_zip_set = set(san_diego_zips_standardized)

Loaded 165 zips from TemporalCoverage_Matrix.csv


**Variable defintions**
- population counts (total + by age group), from B01001 (Sex by Age)
- B01001 male/female age-bracket variables, used to build broad age bucket
- (Under 18 / 18-64 / 65+) rather than pulling ~45 individual brackets.
- Household composition, from B11001 (Household Type) and B11003
- (Family Type by Presence of Own Children) ---
- Median housing costs, from B25064 (Median Gross Rent) and B25088
- (Median Monthly Housing Costs for Owner-Occupied Units) ---
- Poverty rate, from S1701 (Poverty Status in the Past 12 Months) ---
- Unemployment rate, from S2301 (Employment Status) ---

**Population Counts**

In [8]:
POPULATION_VARS = {
    "B01003_001E": "Total_Population",
}

**Male/Female age brackets**
- B01001 male/female age-bracket variables, used to build broad age buckets
- (Under 18 / 18-64 / 65+) rather than pulling ~45 individual brackets.

In [9]:
AGE_MALE_UNDER18 = ["B01001_003E", "B01001_004E", "B01001_005E", "B01001_006E"]
AGE_MALE_18_64 = [f"B01001_{i:03d}E" for i in range(7, 20)]
AGE_MALE_65PLUS = [f"B01001_{i:03d}E" for i in range(20, 26)]
AGE_FEMALE_UNDER18 = ["B01001_027E", "B01001_028E", "B01001_029E", "B01001_030E"]
AGE_FEMALE_18_64 = [f"B01001_{i:03d}E" for i in range(31, 44)]
AGE_FEMALE_65PLUS = [f"B01001_{i:03d}E" for i in range(44, 50)]

AGE_VARS = (
    AGE_MALE_UNDER18 + AGE_MALE_18_64 + AGE_MALE_65PLUS + AGE_FEMALE_UNDER18 + AGE_FEMALE_18_64 + AGE_FEMALE_65PLUS
)

**Household Composition from B11001 (Household Type) and B11003 Family Type by Presence of Own Children**

In [10]:
HOUSEHOLD_VARS = {
    "B11001_001E": "Total_Households",
    "B11001_002E": "Family_Households",
    "B11001_007E": "Nonfamily_Households",
    "B11001_008E": "Householder_Living_Alone",
    "B11003_010E": "Single_Father_Households_With_Own_Children_Under18",
    "B11003_016E": "Single_Mother_Households_With_Own_Children_Under18",

}

**Median Housing Costs**
- Median housing costs, from B25064 (Median Gross Rent) and B25088
- (Median Monthly Housing Costs for Owner-Occupied Units)

In [11]:
HOUSING_VARS = {
    "B25064_001E": "Median_Gross_Rent",
    "B25088_002E": "Median_Monthly_Owner_Cost_With_Mortgage",
    "B25088_003E": "Median_Monthly_Owner_Cost_No_Mortgage",
}

In [12]:
DETAIL_TABLE_VARS = {**POPULATION_VARS, **HOUSEHOLD_VARS, **HOUSING_VARS}
DETAIL_TABLE_FIELDS = list(DETAIL_TABLE_VARS.keys()) + AGE_VARS

**Poverty Rate and unemployment rate**
- Poverty rate, from S1701 (Poverty Status in the Past 12 Months) ---
- Unemployment rate, from S2301 (Employment Status) ---

In [13]:
SUBJECT_TABLE_VARS = {
    "S1701_C03_001E": "Poverty_Rate_Pct",
    "S2301_C04_001E": "Unemployment_Rate_Pct",
}

SUBJECT_TABLE_FIELDS = list(SUBJECT_TABLE_VARS.keys())

**Census Table Table Documentation**

In [14]:
CENSUS_TABLE_DOCS = [
    {"Variable Code": "B01003_001E", "Census Table": "B01003", "Table Name": "Total Population", "Renamed Column": "Total_Population"},
    {"Variable Code": "B01001_003E\u2013006E, 027E\u2013030E", "Census Table": "B01001", "Table Name": "Sex by Age", "Renamed Column": "Population_Under_18"},
    {"Variable Code": "B01001_007E\u2013019E, 031E\u2013043E", "Census Table": "B01001", "Table Name": "Sex by Age", "Renamed Column": "Population_18_to_64"},
    {"Variable Code": "B01001_020E\u2013025E, 044E\u2013049E", "Census Table": "B01001", "Table Name": "Sex by Age", "Renamed Column": "Population_65_Plus"},
    {"Variable Code": "B11001_001E", "Census Table": "B11001", "Table Name": "Household Type (Including Living Alone)", "Renamed Column": "Total_Households"},
    {"Variable Code": "B11001_002E", "Census Table": "B11001", "Table Name": "Household Type (Including Living Alone)", "Renamed Column": "Family_Households"},
    {"Variable Code": "B11001_007E", "Census Table": "B11001", "Table Name": "Household Type (Including Living Alone)", "Renamed Column": "Nonfamily_Households"},
    {"Variable Code": "B11001_008E", "Census Table": "B11001", "Table Name": "Household Type (Including Living Alone)", "Renamed Column": "Householder_Living_Alone"},
    {"Variable Code": "B11003_010E", "Census Table": "B11003", "Table Name": "Family Type by Presence and Age of Own Children Under 18", "Renamed Column": "Single_Father_Households_With_Own_Children_Under18"},
    {"Variable Code": "B11003_016E", "Census Table": "B11003", "Table Name": "Family Type by Presence and Age of Own Children Under 18", "Renamed Column": "Single_Mother_Households_With_Own_Children_Under18"},
    {"Variable Code": "B25064_001E", "Census Table": "B25064", "Table Name": "Median Gross Rent", "Renamed Column": "Median_Gross_Rent"},
    {"Variable Code": "B25088_002E", "Census Table": "B25088", "Table Name": "Median Monthly Housing Costs (Owner-Occupied)", "Renamed Column": "Median_Monthly_Owner_Cost_With_Mortgage"},
    {"Variable Code": "B25088_003E", "Census Table": "B25088", "Table Name": "Median Monthly Housing Costs (Owner-Occupied)", "Renamed Column": "Median_Monthly_Owner_Cost_No_Mortgage"},
    {"Variable Code": "S1701_C03_001E", "Census Table": "S1701", "Table Name": "Poverty Status in the Past 12 Months", "Renamed Column": "Poverty_Rate_Pct"},
    {"Variable Code": "S2301_C04_001E", "Census Table": "S2301", "Table Name": "Employment Status", "Renamed Column": "Unemployment_Rate_Pct"},
]

census_table_docs_df = pd.DataFrame(CENSUS_TABLE_DOCS)
census_table_docs_df

,Variable Code,Census Table,Table Name,Renamed Column
0,B01003_001E,B01003,Total Population,Total_Population
1,"B01001_003E–006E, 027E–030E",B01001,Sex by Age,Population_Under_18
2,"B01001_007E–019E, 031E–043E",B01001,Sex by Age,Population_18_to_64
3,"B01001_020E–025E, 044E–049E",B01001,Sex by Age,Population_65_Plus
4,B11001_001E,B11001,Household Type (Including Living Alone),Total_Households
5,B11001_002E,B11001,Household Type (Including Living Alone),Family_Households
6,B11001_007E,B11001,Household Type (Including Living Alone),Nonfamily_Households
7,B11001_008E,B11001,Household Type (Including Living Alone),Householder_Living_Alone
8,B11003_010E,B11003,Family Type by Presence and Age of Own Childre...,Single_Father_Households_With_Own_Children_Und...
9,B11003_016E,B11003,Family Type by Presence and Age of Own Childre...,Single_Mother_Households_With_Own_Children_Und...


**Pull One Ac5 Vintage**

In [15]:
def _fetch_with_retries(fetch_fn, label, max_attempts=5, backoff_seconds=5):
    """Retry a Census API call a few times on transient network errors
    (dropped connections, incomplete reads) before giving up. Does NOT
    retry on CensusException (e.g. unsupported geography) -- that's a
    real, persistent error, not a network blip, so retrying wastes time.
    """
    from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fetch_fn()
        except (ChunkedEncodingError, ConnectionError, Timeout) as e:
            last_error = e
            print(f"    Network error on {label} (attempt {attempt}/{max_attempts}): {e}")
            if attempt < max_attempts:
                time.sleep(backoff_seconds)
    raise last_error

In [16]:
def pull_vintage(vintage_year):
    """Pull one ACS5 vintage (5-yr window ending in vintage_year), scoped to
    just the San Diego ZCTAs (not all CA ZCTAs) to keep each response small
    and reduce exposure to mid-stream connection resets on flaky networks.
    Returns a DataFrame with one row per matched ZCTA, tagged with the
    vintage's start/end years.
    """
    c = Census(CENSUS_API_KEY, year=vintage_year)

    # Comma-joined list of specific ZCTAs instead of "*" (all ~1,769 CA
    # ZCTAs) -- the Census API accepts a comma-separated value list for
    # "for=zip code tabulation area:...", so this returns only the ~165
    # zips we actually need instead of the whole state.
    zcta_query = ",".join(sorted(sd_zip_set))

    detail_raw = _fetch_with_retries(
        lambda: c.acs5.state_zipcode(
            fields=DETAIL_TABLE_FIELDS, state_fips=STATE_FIPS, zcta=zcta_query, year=vintage_year
        ),
        label=f"detail-table pull ({vintage_year})",
    )
    detail_df = pd.DataFrame(detail_raw)

    subject_table_available = True
    try:
        subject_raw = _fetch_with_retries(
            lambda: c.acs5st.state_zipcode(
                fields=SUBJECT_TABLE_FIELDS, state_fips=STATE_FIPS, zcta=zcta_query, year=vintage_year
            ),
            label=f"subject-table pull ({vintage_year})",
        )
        subject_df = pd.DataFrame(subject_raw)
        subject_df = subject_df.rename(columns={"zip code tabulation area": "Zip Code"})
    except Exception as e:
        print(f"  WARNING: subject-table (poverty/unemployment) pull failed for "
              f"vintage {vintage_year}: {e}\n  Continuing with nulls for this vintage's "
              f"Poverty_Rate_Pct / Unemployment_Rate_Pct.")
        subject_table_available = False
        subject_df = pd.DataFrame({"Zip Code": detail_df["zip code tabulation area"]})
        for col in SUBJECT_TABLE_FIELDS:
            subject_df[col] = np.nan

    detail_df = detail_df.rename(columns={"zip code tabulation area": "Zip Code"})

    # numeric coercion — ACS API returns strings, and suppressed cells return None
    for col in DETAIL_TABLE_FIELDS:
        detail_df[col] = pd.to_numeric(detail_df[col], errors="coerce")
    for col in SUBJECT_TABLE_FIELDS:
        subject_df[col] = pd.to_numeric(subject_df[col], errors="coerce")

    detail_df["Population_Under_18"] = detail_df[AGE_MALE_UNDER18 + AGE_FEMALE_UNDER18].sum(axis=1)
    detail_df["Population_18_to_64"] = detail_df[AGE_MALE_18_64 + AGE_FEMALE_18_64].sum(axis=1)
    detail_df["Population_65_Plus"] = detail_df[AGE_MALE_65PLUS + AGE_FEMALE_65PLUS].sum(axis=1)

    detail_df = detail_df.rename(columns=DETAIL_TABLE_VARS)
    subject_df = subject_df.rename(columns=SUBJECT_TABLE_VARS)

    keep_detail_cols = ["Zip Code"] + list(DETAIL_TABLE_VARS.values()) + [
        "Population_Under_18", "Population_18_to_64", "Population_65_Plus"
    ]
    keep_subject_cols = ["Zip Code"] + list(SUBJECT_TABLE_VARS.values())

    detail_df = detail_df[keep_detail_cols]
    subject_df = subject_df[keep_subject_cols]

    vdf = detail_df.merge(subject_df, on="Zip Code", how="outer")

    int_cols = [
        "Total_Population", "Total_Households", "Family_Households",
        "Nonfamily_Households", "Householder_Living_Alone",
        "Single_Father_Households_With_Own_Children_Under18",
        "Single_Mother_Households_With_Own_Children_Under18",
        "Median_Gross_Rent", "Median_Monthly_Owner_Cost_With_Mortgage",
        "Median_Monthly_Owner_Cost_No_Mortgage",
        "Population_Under_18", "Population_18_to_64", "Population_65_Plus", "Median_Gross_Rent",
        "Median_Monthly_Owner_Cost_With_Mortgage", "Median_Monthly_Owner_Cost_No_Mortgage" 
    ]
    vdf = vdf.astype({c: int for c in int_cols})
    
    vdf = vdf.replace([-666666666], np.nan)

    vdf["ACS_Vintage_Start_Year"] = vintage_year - 4
    vdf["ACS_Vintage_End_Year"] = vintage_year
    vdf["Subject_Table_Available"] = subject_table_available
    return vdf

**Pull every published vintage**

One pull per calendar year 2018-2024 (14 API calls total: detail + subject
table per year). 2025 and 2026 have no published vintage yet, so they are
not pulled here — they get mapped to the latest available vintage in the
next step.

In [17]:
published_years = [y for y in DATA_YEARS if y <= LATEST_PUBLISHED_VINTAGE]
vintage_frames = {}
failed_years = []

for y in published_years:
    print(f"Pulling ACS5 vintage ending {y} ({y-4}-{y})...")
    try:
        vintage_frames[y] = pull_vintage(y)
    except Exception as e:
        print(f"  FAILED to pull vintage {y} after retries: {e}\n  Skipping this year "
              f"— it will fall back to the nearest available vintage below.")
        failed_years.append(y)
    time.sleep(0.5)  # be polite to the API

print(f"\nPulled {len(vintage_frames)} vintages: {sorted(vintage_frames.keys())}")
if failed_years:
    print(f"Years that failed entirely and will use a fallback vintage instead: {failed_years}")

Pulling ACS5 vintage ending 2018 (2014-2018)...
Pulling ACS5 vintage ending 2019 (2015-2019)...
  Continuing with nulls for this vintage's Poverty_Rate_Pct / Unemployment_Rate_Pct.
Pulling ACS5 vintage ending 2020 (2016-2020)...
Pulling ACS5 vintage ending 2021 (2017-2021)...
Pulling ACS5 vintage ending 2022 (2018-2022)...
Pulling ACS5 vintage ending 2023 (2019-2023)...
Pulling ACS5 vintage ending 2024 (2020-2024)...

Pulled 7 vintages: [2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [18]:
year_to_vintage = {}

for y in DATA_YEARS:
    year_to_vintage[y] = y if y in vintage_frames else LATEST_PUBLISHED_VINTAGE

print("Data year -> ACS vintage used:")
for y in DATA_YEARS:
    fallback_note = "  (fallback — no published vintage for this year)" if year_to_vintage[y] != y else ""
    print(f"  {y} -> {year_to_vintage[y]-4}-{year_to_vintage[y]}{fallback_note}")

all_years = []
for y in DATA_YEARS:
    vdf = vintage_frames[year_to_vintage[y]].copy()
    vdf["Data_Year"] = y
    all_years.append(vdf)

acs_all_years = pd.concat(all_years, ignore_index=True)
print(f"\n{acs_all_years.shape[0]} zip-year rows across {acs_all_years['Zip Code'].nunique()} ZCTAs and {len(DATA_YEARS)} years")

Data year -> ACS vintage used:
  2018 -> 2014-2018
  2019 -> 2015-2019
  2020 -> 2016-2020
  2021 -> 2017-2021
  2022 -> 2018-2022
  2023 -> 2019-2023
  2024 -> 2020-2024
  2025 -> 2020-2024  (fallback — no published vintage for this year)
  2026 -> 2020-2024  (fallback — no published vintage for this year)

927 zip-year rows across 104 ZCTAs and 9 years


In [19]:
acs_sd = acs_all_years[acs_all_years["Zip Code"].isin(san_diego_zips_standardized)].copy()
acs_sd = acs_sd.sort_values(["Zip Code", "Data_Year"]).reset_index(drop=True)

matched_zips = set(acs_sd["Zip Code"])
unmatched_zips = sorted(sd_zip_set - matched_zips)

print(f"Matched {len(matched_zips)} / {len(sd_zip_set)} San Diego zips to a ZCTA (across all years).")
if unmatched_zips:
    print(f"Unmatched zips (no ZCTA data returned in any vintage): {unmatched_zips}")

Matched 104 / 165 San Diego zips to a ZCTA (across all years).
Unmatched zips (no ZCTA data returned in any vintage): ['91903', '91908', '91909', '91912', '91921', '91933', '91943', '91944', '91946', '91951', '91976', '91979', '92013', '92018', '92022', '92023', '92030', '92033', '92038', '92039', '92046', '92049', '92051', '92052', '92068', '92072', '92074', '92079', '92085', '92088', '92112', '92137', '92138', '92142', '92143', '92149', '92150', '92153', '92159', '92160', '92163', '92165', '92166', '92167', '92168', '92169', '92170', '92171', '92172', '92174', '92175', '92176', '92177', '92178', '92186', '92191', '92192', '92193', '92195', '92196', '92198']


In [20]:
print(acs_sd.isna().sum())
print()

Zip Code                                                0
Total_Population                                        0
Total_Households                                        0
Family_Households                                       0
Nonfamily_Households                                    0
Householder_Living_Alone                                0
Single_Father_Households_With_Own_Children_Under18      0
Single_Mother_Households_With_Own_Children_Under18      0
Median_Gross_Rent                                     103
Median_Monthly_Owner_Cost_With_Mortgage                69
Median_Monthly_Owner_Cost_No_Mortgage                  96
Population_Under_18                                     0
Population_18_to_64                                     0
Population_65_Plus                                      0
Poverty_Rate_Pct                                      136
Unemployment_Rate_Pct                                 105
ACS_Vintage_Start_Year                                  0
ACS_Vintage_En

In [21]:
(acs_sd.isna().sum() / len(acs_sd)) * 100

Zip Code                                               0.000000
Total_Population                                       0.000000
Total_Households                                       0.000000
Family_Households                                      0.000000
Nonfamily_Households                                   0.000000
Householder_Living_Alone                               0.000000
Single_Father_Households_With_Own_Children_Under18     0.000000
Single_Mother_Households_With_Own_Children_Under18     0.000000
Median_Gross_Rent                                     11.111111
Median_Monthly_Owner_Cost_With_Mortgage                7.443366
Median_Monthly_Owner_Cost_No_Mortgage                 10.355987
Population_Under_18                                    0.000000
Population_18_to_64                                    0.000000
Population_65_Plus                                     0.000000
Poverty_Rate_Pct                                      14.670982
Unemployment_Rate_Pct                   

In [22]:
demo_columns = [c for c in acs_sd.columns if c not in (
    "Zip Code", "Data_Year", "ACS_Vintage_Start_Year", "ACS_Vintage_End_Year", "Is_Fallback_Vintage"
)]

acs_sd["Has_Missing_Demographic_Data"] = acs_sd[demo_columns].isna().any(axis=1)

In [24]:
sd_zip_strs = set(str(z).zfill(5) for z in san_diego_zips_standardized)

In [25]:
acs_sd["Zip_In_San_Diego_County"] = acs_sd["Zip Code"].isin(sd_zip_strs)

In [26]:
acs_in_sd = (
    acs_sd[acs_sd["Zip_In_San_Diego_County"]]
    .drop(columns="Zip_In_San_Diego_County")
    .copy()
)

missing_or_invalid = (
    acs_sd[~acs_sd["Zip_In_San_Diego_County"]]
    .drop(columns="Zip_In_San_Diego_County")
    .copy()
)

In [27]:
missing_or_invalid.head(5)

,Zip Code,Total_Population,Total_Households,Family_Households,Nonfamily_Households,Householder_Living_Alone,Single_Father_Households_With_Own_Children_Under18,Single_Mother_Households_With_Own_Children_Under18,Median_Gross_Rent,Median_Monthly_Owner_Cost_With_Mortgage,...,Population_Under_18,Population_18_to_64,Population_65_Plus,Poverty_Rate_Pct,Unemployment_Rate_Pct,ACS_Vintage_Start_Year,ACS_Vintage_End_Year,Subject_Table_Available,Data_Year,Has_Missing_Demographic_Data


In [28]:
missing = acs_in_sd[acs_in_sd["Has_Missing_Demographic_Data"]].copy()

missing["Missing_Fields"] = missing[demo_columns].apply(
    lambda row: ", ".join(row[row.isna()].index), axis=1
)

missing["Missing_Field_Count"] = missing[demo_columns].isna().sum(axis=1)

In [29]:
missingness_by_zip = missing[['Zip Code', 'Missing_Field_Count']] \
    .sort_values('Missing_Field_Count', ascending=False) \
    .reset_index(drop=True)

In [30]:
acs_in_sd.to_csv("Demographics_SD_ZipCode.csv", index=False)
print(f"\nSaved ACS5_ZCTA_SanDiego.csv — {acs_sd.shape[0]} rows x {acs_sd.shape[1]} columns")
print(acs_sd.head())


Saved ACS5_ZCTA_SanDiego.csv — 927 rows x 22 columns
  Zip Code  Total_Population  Total_Households  Family_Households  \
0    91901             18558              6411               4683   
1    91901             18162              6320               4576   
2    91901             17896              6150               4584   
3    91901             18367              6313               4766   
4    91901             18669              6425               4793   

   Nonfamily_Households  Householder_Living_Alone  \
0                  1728                      1539   
1                  1744                      1620   
2                  1566                      1448   
3                  1547                      1335   
4                  1632                      1241   

   Single_Father_Households_With_Own_Children_Under18  \
0                                                137    
1                                                196    
2                                        

In [31]:
census_table_docs_df.to_csv("Census_Table_Documentation.csv", index=False)
print(f"\nSaved Census_Table_Documentation.csv — {census_table_docs_df.shape[0]} variables documented")


Saved Census_Table_Documentation.csv — 15 variables documented
